# Подготовка ToolBench G1, G2 и G3

Запускай из папки `data/ToolBench`. Каждая выборка использует свои ID и сохраняется в своей папке. Источники исходных файлов: [Tool-COLT/ToolBenchG2](https://huggingface.co/datasets/Tool-COLT/ToolBenchG2/tree/8f5e03f6e345bc5c9372060a92e7490d6cdf6d34) и [Tool-COLT/ToolBenchG3](https://huggingface.co/datasets/Tool-COLT/ToolBenchG3/tree/c408d83703150aec216903c0c2e83c5ebfcbe4e7).

G1 взята из [Adorg/ToolBench](https://huggingface.co/datasets/Adorg/ToolBench/tree/0c25df5e6937aae854c2cc1bb203297600afce8b/retrieval/G1). Файлы запросов и связей train/test объединяются перед разбиением 80/10/10 с SEED = 67, как в G2/G3. Для description берём полный текст document_content (JSON с документацией API).


In [1]:
import json
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 67

In [ ]:
def check_empty(table, field):
    if field not in table:
        print(f"{field}: столбец отсутствует")
        return
    empty = table[field].map(lambda x: x is None or x == {} or x == [] or
                             (isinstance(x, str) and not x.strip()))
    print(f"{field}: пустых {empty.sum()} из {len(table)}")


def load_g1(raw_dir):
    query_tables = []
    label_tables = []
    for split in ["train", "test"]:
        query_tables.append(pd.read_csv(
            raw_dir / f"{split}.query.txt", sep="\t", names=["_id", "text"],
            dtype=str, keep_default_na=False,
        ))
        labels = pd.read_csv(
            raw_dir / f"qrels.{split}.tsv", sep="\t", header=None,
            names=["Request_id", "unused", "API_id", "label"], dtype="int64",
        )
        assert labels["label"].eq(1).all(), "Проверь значения relevance в G1"
        label_tables.append(labels[["Request_id", "API_id"]])

    queries = pd.concat(query_tables, ignore_index=True)
    interactions = pd.concat(label_tables, ignore_index=True)
    apis = pd.read_csv(raw_dir / "corpus.tsv", sep="\t", dtype=str, keep_default_na=False)
    apis = apis.rename(columns={"docid": "_id", "document_content": "text"})

    # Один и тот же запрос или API записан в TSV несколько раз.
    # Убираем только полные повторы. Конфликты ID проверяются в process_data.
    print("G1 — повторных строк запросов:", queries.duplicated().sum())
    print("G1 — повторных строк каталога API:", apis.duplicated().sum())
    queries = queries.drop_duplicates().reset_index(drop=True)
    apis = apis.drop_duplicates().reset_index(drop=True)

    docs = pd.DataFrame(apis["text"].map(json.loads).tolist())
    print("G1 — поля внутри document_content")
    for field in ["metadata", "title"]:
        check_empty(docs, field)
    empty_descriptions = docs["api_description"].fillna("").str.strip().eq("").sum()
    print("Пустых api_description:", empty_descriptions, "из", len(docs))

    apis["text"] = apis["text"].str.replace(
    r"ghp_[A-Za-z0-9]{36}|github_pat_[A-Za-z0-9_]{22,}",
    "[REDACTED]",
    regex=True,
    )

    return queries, apis, interactions


def process_data(group):
    raw_dir = Path("raw") / group
    output_dir = Path("processed") / group

    if group == "G1":
        queries, apis, interactions = load_g1(raw_dir)
    else:
        queries = pd.read_json(raw_dir / "queries.jsonl", lines=True, dtype={"_id": str})
        apis = pd.read_json(raw_dir / "corpus.jsonl", lines=True, dtype={"_id": str})
        interactions = pd.read_csv(raw_dir / "query_tool.txt", sep="\t",
                                   names=["Request_id", "API_id"], dtype="int64")

    print(group, "— поля запросов")
    check_empty(queries, "metadata")
    print(group, "— поля API")
    check_empty(apis, "metadata")
    check_empty(apis, "title")

    request_ids = queries["_id"].astype("int64")
    api_ids = apis["_id"].astype("int64")
    assert request_ids.is_unique and request_ids.ge(0).all(), "Проверь Request_id"
    assert api_ids.is_unique and api_ids.ge(0).all(), "Проверь API_id"
    print("Request_id и API_id уникальны внутри выборки")

    request_id_map = pd.DataFrame({"source_id": queries["_id"], "Request_id": request_ids})
    api_id_map = pd.DataFrame({"source_id": apis["_id"], "API_id": api_ids})
    queries = pd.DataFrame({"Request_id": request_ids, "text": queries["text"]})
    apis = pd.DataFrame({"API_id": api_ids, "description": apis["text"]})
    assert queries["text"].fillna("").str.strip().ne("").all()
    assert apis["description"].fillna("").str.strip().ne("").all()
    assert queries["text"].is_unique, "Есть точные дубликаты запросов"

    # Одинаковые после нормализации тексты должны попасть в один split.
    texts = queries["text"].str.strip().str.casefold()
    print("Повторных текстов после strip и casefold:", texts.duplicated().sum())
    train_texts, remaining_texts = train_test_split(
        texts.drop_duplicates(), test_size=0.2, random_state=SEED
    )
    val_texts, test_texts = train_test_split(
        remaining_texts, test_size=0.5, random_state=SEED
    )
    queries["split"] = texts.map({
        **dict.fromkeys(train_texts, "train"),
        **dict.fromkeys(val_texts, "val"),
        **dict.fromkeys(test_texts, "test"),
    })
    print(queries["split"].value_counts())

    assert interactions["Request_id"].isin(request_ids).all()
    assert interactions["API_id"].isin(api_ids).all()
    print("Повторных пар:", interactions.duplicated(["Request_id", "API_id"]).sum())
    interactions = interactions.drop_duplicates(["Request_id", "API_id"]).reset_index(drop=True)
    interactions["relevance"] = 1.0

    output_dir.mkdir(parents=True, exist_ok=True)
    queries.to_parquet(output_dir / "requests.parquet", index=False)
    apis.to_parquet(output_dir / "apis.parquet", index=False)
    interactions.to_parquet(output_dir / "interactions.parquet", index=False)
    maps_dir = output_dir / "id_maps"
    maps_dir.mkdir(exist_ok=True)
    request_id_map.to_parquet(maps_dir / "requests.parquet", index=False)
    api_id_map.to_parquet(maps_dir / "apis.parquet", index=False)
    print(f"Запросов: {len(queries)}, API: {len(apis)}, связей: {len(interactions)}")

In [3]:
process_data("G2")

G2 — поля запросов
metadata: пустых 82507 из 82507
G2 — поля API
metadata: пустых 11474 из 11474
title: пустых 11474 из 11474
Request_id и API_id уникальны внутри выборки
Повторных текстов после strip и casefold: 1
split
train    66005
val       8251
test      8251
Name: count, dtype: int64
Повторных пар: 201
Запросов: 82507, API: 11474, связей: 208143


In [4]:
process_data("G3")

G3 — поля запросов
metadata: пустых 23734 из 23734
G3 — поля API
metadata: пустых 1419 из 1419
title: пустых 1419 из 1419
Request_id и API_id уникальны внутри выборки
Повторных текстов после strip и casefold: 2
split
train    18986
test      2375
val       2373
Name: count, dtype: int64
Повторных пар: 66
Запросов: 23734, API: 1419, связей: 64295


In [5]:
process_data("G1")

G1 — повторных строк запросов: 345426
G1 — повторных строк каталога API: 422360
G1 — поля внутри document_content
metadata: столбец отсутствует
title: столбец отсутствует
Пустых api_description: 789 из 10439
G1 — поля запросов
metadata: столбец отсутствует
G1 — поля API
metadata: столбец отсутствует
title: столбец отсутствует
Request_id и API_id уникальны внутри выборки
Повторных текстов после strip и casefold: 0
split
train    69935
test      8742
val       8742
Name: count, dtype: int64
Повторных пар: 8676
Запросов: 87419, API: 10439, связей: 424169
